# MTL inference test: semantic segmentation + depth (Cityscapes)

**Osama Yehya Ghazal** · NIM 26/591650/PPA/07466 · MII226401 Computational Intelligence and Machine Learning, Class B252

Lecturer: Oskar Natan, S.ST., M.Tr.T., Ph.D. · Assignment: MTL Computer Vision

This notebook does the inference test only (train-val were optional, so I skipped them), plus a few extra inference-only experiments at the end. It uses the trained model from the course folder, `trained_model/best_model.pth`, and the code from `mtl_seg_dep_colab.ipynb`. The helper functions, the model and the dataset class are copied from the lecturer's notebook. I only changed the paths (local folder instead of Google Drive) and I run it on CPU. I tried to get the full Cityscapes data too, but the download needs an account that is approved by hand and mine was not approved yet, so the main results use the 15 images in the course folder. For a bigger segmentation test I also use 1,701 public images from CamVid and comma10k.

The `cityscapes/` and `trained_model/` folders from the course Drive need to be next to this notebook (and `resunet_mtl_0/` for the 2-epoch comparison). CamVid and comma10k are downloaded with git the first time if they are not there (about 850 MB together); these two parts take the longest to run on a CPU.

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# disp2dep and logdepth divide by zero where there is no disparity; that's expected, so I hide the warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()  # the folder this notebook is in
dataset_dir = str(ROOT / "cityscapes") + os.sep
model_path = ROOT / "trained_model" / "best_model.pth"
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

assert os.path.isdir(dataset_dir), "cityscapes folder not found next to the notebook"
assert model_path.exists(), "trained_model/best_model.pth not found next to the notebook"

device = torch.device("cpu")
print("torch", torch.__version__, "| device:", device)

## Helper functions (from the lecturer's notebook)
Label map (35 Cityscapes IDs to 20 classes), one-hot encoding, disparity to depth, resizing and the dataset class, copied as they are.

In [ ]:
from torch.utils.data import Dataset

#Define colormap dan konversi label dari 35 class ke 20 class
#BACA https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py
SEG_MAP = {
    'colors'        :[[0, 0, 0], [128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156],
                    [190, 153, 153], [153, 153, 153], [250, 170, 30], [220, 220, 0],
                    [107, 142, 35], [152, 251, 152], [70, 130, 180], [220, 20, 60],
                    [255, 0, 0], [0, 0, 142], [0, 0, 70], [0, 60, 100],
                    [0, 80, 100], [0, 0, 230], [119, 11, 32]],
    'classes'       : ['None', 'road', 'sidewalk', 'building', 'wall',
                        'fence', 'pole', 'traffic light', 'traffic sign',
                        'vegetation', 'terrain', 'sky', 'person',
                        'rider', 'car', 'truck', 'bus',
                        'train', 'motorcycle', 'bicycle'],
}

mapping_20 = {
        0: 0,
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 0,
        6: 0,
        7: 1,
        8: 2,
        9: 0,
        10: 0,
        11: 3,
        12: 4,
        13: 5,
        14: 0,
        15: 0,
        16: 0,
        17: 6,
        18: 0,
        19: 7,
        20: 8,
        21: 9,
        22: 10,
        23: 11,
        24: 12,
        25: 13,
        26: 14,
        27: 15,
        28: 16,
        29: 0,
        30: 0,
        31: 17,
        32: 18,
        33: 19,
        -1: 0
    }


#konversi dari 35 class ke 20 class
def encode_labels(mask, mapping_label):
    #inputnya adalah HWC hasil baca opencv
    mask = mask[:,:,0:1].reshape(mask.shape[0], mask.shape[1])  #ambil 1 channel dan reshape dari 3D ke 2D
    label_mask = np.zeros_like(mask)
    for k in mapping_label:
        label_mask[mask == k] = mapping_label[k]
    return label_mask

#konversi dari class id ke representasi one-hot encoding
def cls2one_hot(ss_gt20, n_class):
    one_hot = (np.arange(n_class) == ss_gt20[...,None]).astype(int)
    return one_hot

#konversi disparity ke depth
#baca https://github.com/mcordts/cityscapesScripts/blob/master/README.md#dataset-structure
def disp2dep(img_d, dep_max):
    #d = ( float(p) - 1. ) / 256
    img_d[img_d > 0] = (img_d[img_d > 0] - 1) / 256
    #depth = baseline * focal length / disparity
    depth = (0.209313 * 2262.52) / img_d
    #clip depth maximum ke 100 meter, karena langit nilai depthnya terlalu tinggi
    depth = np.minimum(depth, dep_max)
    return depth

#untuk mereduksi dimensi spasial supaya komputasi tidak terlalu berat
def resize_matrix(image, resize):
    #inputnya HWC
    WH_resized = (int(image.shape[1]/resize), int(image.shape[0]/resize))
    resized_image = cv2.resize(image, WH_resized, interpolation=cv2.INTER_NEAREST)

    return resized_image


#buat class dataset
class myData(Dataset):
    def __init__(self, root_dir, part):
        #buat list kosong dulu
        self.rgb = []
        self.seg = []
        self.dep = []

        #define folder masing2 untuk rgb seg dep
        rgb_folder = os.path.join(os.path.join(os.path.join(root_dir, 'leftImg8bit_trainvaltest'), 'leftImg8bit'), part)
        seg_folder = os.path.join(os.path.join(os.path.join(root_dir, 'gtFine_trainvaltest'), 'gtFine'), part)
        dep_folder = os.path.join(os.path.join(os.path.join(root_dir, 'disparity_trainvaltest'), 'disparity'), part)

        # list semua kota yang ada di RGB leftimg8bit
        town_list = os.listdir(rgb_folder)
        town_list.sort()
        for town in town_list:
            rgb_dir = os.path.join(rgb_folder, town)
            seg_dir = os.path.join(seg_folder, town)
            dep_dir = os.path.join(dep_folder, town)

            #list semua file
            file_list = os.listdir(rgb_dir)
            file_list.sort()
            for filename in file_list:
                #append ke list kosong tadi
                self.rgb.append(os.path.join(rgb_dir, filename))
                self.seg.append(os.path.join(seg_dir, filename[:-15]+'gtFine_labelIds.png'))
                self.dep.append(os.path.join(dep_dir, filename[:-15]+'disparity.png'))

    def __len__(self):
        return len(self.rgb)

    def __getitem__(self, index):
        data = dict()
        data['rgb'] = np.transpose(np.array(resize_matrix(
            cv2.imread(self.rgb[index]),
            resize=4)), (2, 0, 1)) # (H, W, C) --> (C, H, W)
        data['seg'] = np.transpose(np.array(resize_matrix(cls2one_hot(encode_labels(
            cv2.imread(self.seg[index]),
            mapping_label=mapping_20), n_class=len(SEG_MAP['classes'])), resize=4)), (2, 0, 1)) # (H, W, C) --> (C, H, W)
        data['dep'] = np.expand_dims(np.array(resize_matrix(disp2dep(
            cv2.imread(self.dep[index], cv2.IMREAD_UNCHANGED).astype(np.float32),
            dep_max=100), resize=4)), axis=0) / 100 #range dinormalisasi ke 0 - 1

        return data


In [ ]:
#visualisasi segmentation
def vis_seg(seg, SEG_CMAP):
    #buat array kosong untuk menyimpan output gambar
    imgx = np.zeros((seg.shape[0], seg.shape[1], 3))
    #warnai setiap label
    for cmap in SEG_CMAP['colors']:
        cmap_id = SEG_CMAP['colors'].index(cmap)
        imgx[np.where(seg == cmap_id)] = cmap
#         imgx = imgx[:, :, [2, 1, 0]]
    return imgx


#untuk visualisasi logarithmic depth
def logdepth(normalized_depth):
    logdepth = np.ones(normalized_depth.shape) + (np.log(normalized_depth) / 5.70378)
    logdepth = np.clip(logdepth, 0.0, 1.0) * 255 #normalisasi ke 0 - 255
    logdepth = np.repeat(logdepth[:, :, np.newaxis], 3, axis=2)
    return logdepth


Quick check that the label mapping is right: the course folder also has `*_gtFine_labelIds_20c.png` files that are already converted to 20 classes. The notebook's `encode_labels` should give exactly the same result.

In [ ]:
import glob
match = []
for f in sorted(glob.glob(dataset_dir + "gtFine_trainvaltest/gtFine/*/*/*_gtFine_labelIds.png")):
    c20_file = f.replace("_labelIds.png", "_labelIds_20c.png")
    if not os.path.exists(c20_file):
        continue  # the test split has no 20-class files
    mine = encode_labels(cv2.imread(f), mapping_20)
    theirs = cv2.imread(c20_file)[:, :, 0]
    match.append((mine == theirs).mean())
print(f"{len(match)} label files checked, pixels matching: {100 * np.mean(match):.2f}%")

## Model: ResUNet-MTL (from the lecturer's notebook)
One ResNet-34 encoder shared by two U-Net decoders, one for segmentation (20 channels, sigmoid) and one for depth (1 channel, ReLU). The only change: `resnet34(weights=None)`, because all the weights come from `best_model.pth` anyway.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch import cat
from torchvision import models
import torchvision.transforms as transforms


class ConvBNRelu(nn.Module):
    def __init__(self, channelx, stridex=1, kernelx=3, paddingx=1):
        super(ConvBNRelu, self).__init__()
        self.conv = nn.Conv2d(channelx[0], channelx[1], kernel_size=kernelx, stride=stridex, padding=paddingx, padding_mode='zeros')
        self.bn = nn.BatchNorm2d(channelx[1])
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        y = self.relu(x)
        return y

class ConvBlock(nn.Module):
    def __init__(self, channel, final=False):
        super(ConvBlock, self).__init__()
        #conv block
        if final:
            self.conv_block0 = ConvBNRelu(channelx=[channel[0], channel[0]], stridex=1)
            self.conv_block1 = nn.Sequential(
            nn.Conv2d(channel[0], channel[1], kernel_size=1)
            )
        else:
            self.conv_block0 = ConvBNRelu(channelx=[channel[0], channel[1]], stridex=1)
            self.conv_block1 = ConvBNRelu(channelx=[channel[1], channel[1]], stridex=1)


    def forward(self, x):
        #convolutional block
        y = self.conv_block0(x)
        y = self.conv_block1(y)
        return y


class resunet_mtl(nn.Module): #
    def __init__(self):
        super().__init__()
        n_fmap = [64, 64, 128, 256, 512]

        #RGB
        self.rgb_normalizer = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.RGB_encoder = models.resnet34(weights=None) # bobot diambil dari best_model.pth, jadi tidak perlu download ImageNet #resnet18
        self.RGB_encoder.fc = nn.Sequential() #cara paling gampang untuk menghilangkan fc layer yang tidak diperlukan
        self.RGB_encoder.avgpool = nn.Sequential()

        #SEG
        self.conv3_ss_f = ConvBlock(channel=[n_fmap[4]+n_fmap[3], n_fmap[3]])
        self.conv2_ss_f = ConvBlock(channel=[n_fmap[3]+n_fmap[2], n_fmap[2]])
        self.conv1_ss_f = ConvBlock(channel=[n_fmap[2]+n_fmap[1], n_fmap[1]])
        self.conv0_ss_f = ConvBlock(channel=[n_fmap[1]+n_fmap[0], n_fmap[0]])
        self.final_ss_f = ConvBlock(channel=[n_fmap[0], 20], final=True)

        #DEPTH
        self.conv3_dep_f = ConvBlock(channel=[n_fmap[4]+n_fmap[3], n_fmap[3]])
        self.conv2_dep_f = ConvBlock(channel=[n_fmap[3]+n_fmap[2], n_fmap[2]])
        self.conv1_dep_f = ConvBlock(channel=[n_fmap[2]+n_fmap[1], n_fmap[1]])
        self.conv0_dep_f = ConvBlock(channel=[n_fmap[1]+n_fmap[0], n_fmap[0]])
        self.final_dep_f = ConvBlock(channel=[n_fmap[0], 1], final=True)

        #aktivasi final
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()

    def forward(self, rgb):
        #bagian downsampling
        in_rgb = self.rgb_normalizer(rgb)
        x_RGB = self.RGB_encoder.conv1(in_rgb)
        x_RGB = self.RGB_encoder.bn1(x_RGB)
        RGB_features0 = self.RGB_encoder.relu(x_RGB)
        RGB_features1 = self.RGB_encoder.layer1(self.RGB_encoder.maxpool(RGB_features0))
        RGB_features2 = self.RGB_encoder.layer2(RGB_features1)
        RGB_features3 = self.RGB_encoder.layer3(RGB_features2)
        RGB_features4 = self.RGB_encoder.layer4(RGB_features3)
        up_RGB_features4 = F.interpolate(RGB_features4, size=RGB_features3.size()[-2:], mode='bilinear', align_corners=True)

        #bagian segmentation
        ss_f_3 = self.conv3_ss_f(cat([up_RGB_features4, RGB_features3], dim=1))
        up_ss_f_3 = F.interpolate(ss_f_3, size=RGB_features2.size()[-2:], mode='bilinear', align_corners=True)
        ss_f_2 = self.conv2_ss_f(cat([up_ss_f_3, RGB_features2], dim=1))
        up_ss_f_2 = F.interpolate(ss_f_2, size=RGB_features1.size()[-2:], mode='bilinear', align_corners=True)
        ss_f_1 = self.conv1_ss_f(cat([up_ss_f_2, RGB_features1], dim=1))
        up_ss_f_1 = F.interpolate(ss_f_1, size=RGB_features0.size()[-2:], mode='bilinear', align_corners=True)
        ss_f_0 = self.conv0_ss_f(cat([up_ss_f_1, RGB_features0], dim=1))
        up_ss_f_0 = F.interpolate(ss_f_0, size=in_rgb.size()[-2:], mode='bilinear', align_corners=True)
        ss_f = self.sigmoid(self.final_ss_f(up_ss_f_0))

        #bagian depth estimation
        dep_f_3 = self.conv3_dep_f(cat([up_RGB_features4, RGB_features3], dim=1))
        up_dep_f_3 = F.interpolate(dep_f_3, size=RGB_features2.size()[-2:], mode='bilinear', align_corners=True)
        dep_f_2 = self.conv2_dep_f(cat([up_dep_f_3, RGB_features2], dim=1))
        up_dep_f_2 = F.interpolate(dep_f_2, size=RGB_features1.size()[-2:], mode='bilinear', align_corners=True)
        dep_f_1 = self.conv1_dep_f(cat([up_dep_f_2, RGB_features1], dim=1))
        up_dep_f_1 = F.interpolate(dep_f_1, size=RGB_features0.size()[-2:], mode='bilinear', align_corners=True)
        dep_f_0 = self.conv0_dep_f(cat([up_dep_f_1, RGB_features0], dim=1))
        up_dep_f_0 = F.interpolate(dep_f_0, size=in_rgb.size()[-2:], mode='bilinear', align_corners=True)
        dep_f = self.relu(self.final_dep_f(up_dep_f_0))

        return ss_f, dep_f


## Load the trained model

In [ ]:
model = resunet_mtl()
print(model.load_state_dict(torch.load(model_path, map_location=device)))
model.to(device, dtype=torch.float)
model.eval()  # the original cell 30 is missing this, see the check at the end

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.2f} M")

## D. TEST MODEL: one validation image
Same image the lecturer uses in cells 29 and 30, the first frankfurt image of the val split.

In [ ]:
dir_rgb = dataset_dir + "leftImg8bit_trainvaltest/leftImg8bit/"
dir_seg = dataset_dir + "gtFine_trainvaltest/gtFine/"
dir_dep = dataset_dir + "disparity_trainvaltest/disparity/"
part, town = "val/", "frankfurt/"

filename = sorted(os.listdir(dir_rgb + part + town))[0]
print(filename)
image = cv2.imread(dir_rgb + part + town + filename)
seg_gt = cv2.imread(dir_seg + part + town + filename[:-15] + "gtFine_labelIds.png")
dep_gt = cv2.imread(dir_dep + part + town + filename[:-15] + "disparity.png", cv2.IMREAD_UNCHANGED).astype(np.float32)

# same preprocessing as the dataloader: resize by 4, HWC -> NCHW
input_img = np.expand_dims(resize_matrix(image, resize=4).transpose(2, 0, 1), axis=0)
input_img = torch.from_numpy(input_img).to(device, dtype=torch.float)
print("input image shape", input_img.size())

with torch.no_grad():
    pred_seg, pred_dep = model(input_img)
pred_seg = np.argmax(pred_seg.cpu().numpy()[0], axis=0)
pred_dep = pred_dep.cpu().numpy()[0][0]
print("output shapes", pred_seg.shape, pred_dep.shape)

panels = [image[:, :, [2, 1, 0]],
          vis_seg(encode_labels(seg_gt, mapping_20), SEG_MAP),
          vis_seg(pred_seg, SEG_MAP),
          logdepth(disp2dep(dep_gt, 100) / 100),
          logdepth(pred_dep)]
titles = ["RGB", "GT segmentation", "predicted segmentation", "GT depth (log)", "predicted depth (log)"]
fig, axs = plt.subplots(1, 5, figsize=(18, 2.6))
for ax, im, t in zip(axs, panels, titles):
    ax.imshow(im.astype(np.uint8)); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.savefig(out_dir / "test_model_val0.png", dpi=150); plt.show()

## E. TEST KE SELURUH SAMPLE: whole test split
The metrics are the ones from the notebook: IoU over all 20 one-hot channels (threshold 0.5), MAE on the normalised depth, and TM = (1 - IoU) + MAE (lower is better). I also log the depth error in metres on pixels that have real disparity, because the plain MAE also counts pixels with no depth (those are set to 100 m).

In [ ]:
#FUNGSI IOU (from the lecturer's notebook)
def IOU(Yp, Yt):
    output = Yp.view(-1) > 0.5
    target = Yt.view(-1) > 0.5
    intersection = (output & target).sum()
    union = (output | target).sum()
    return intersection / union


def run_split(net, part, swap_to_rgb=False):
    ds = myData(root_dir=dataset_dir, part=part)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    rows, maps = [], []
    with torch.no_grad():
        for i, data in enumerate(loader):
            image = data["rgb"].to(device, dtype=torch.float)
            seg_gt = data["seg"].to(device, dtype=torch.float)
            dep_gt = data["dep"].to(device, dtype=torch.float)
            if swap_to_rgb:
                image = image[:, [2, 1, 0]]

            t0 = time.time()
            pred_seg, pred_dep = net(image)
            elapsed = time.time() - t0

            iou = IOU(pred_seg, seg_gt).item()
            mae = F.l1_loss(pred_dep, dep_gt).item()
            valid = (dep_gt > 0) & (dep_gt < 1)  # pixels that really have a depth value
            mae_m = ((pred_dep - dep_gt).abs()[valid].mean() * 100).item()

            p, g = pred_seg.argmax(1)[0].numpy(), seg_gt.argmax(1)[0].numpy()
            lab = g > 0
            rows.append({"image": Path(ds.rgb[i]).name[:-16], "IoU": iou, "MAE": mae, "TM": (1 - iou) + mae,
                         "MAE_valid_m": mae_m, "pixel_acc": (p[lab] == g[lab]).mean() if lab.any() else np.nan,
                         "time_s": elapsed})
            maps.append((p, g))
    return pd.DataFrame(rows), maps


test_df, test_maps = run_split(model, "test")
lecturer = pd.read_csv(ROOT / "trained_model" / "test_log.csv")
test_df["TM_lecturer_log"] = lecturer["test_tm"]   # the lecturer's log only has images 0-3
test_df.to_csv(out_dir / "test_log.csv", index=False)
show = test_df.drop(columns="pixel_acc")
display(show.round(4))
print("average:", show.drop(columns=["image", "TM_lecturer_log"]).mean().round(4).to_dict())

The test IoU is very low. At first I thought something was wrong, so I looked at what the test label files actually contain:

In [ ]:
lbl = cv2.imread(dataset_dir + "gtFine_trainvaltest/gtFine/test/berlin/berlin_000000_000019_gtFine_labelIds.png",
                 cv2.IMREAD_UNCHANGED)
print("label IDs in a test label file:", np.unique(lbl))
print("after mapping to 20 classes:", np.unique(encode_labels(lbl[:, :, None], mapping_20)))
for i, (p, g) in enumerate(test_maps):
    print(f"test image {i}: {100 * (p == 0).mean():.1f}% of pixels predicted as 'None'")

So the test labels are just placeholders (Cityscapes does not publish its test labels): IDs 0, 1 and 3 are unlabeled, ego vehicle and out of ROI, and all of them map to 'None'. The test IoU only measures how many pixels the model calls 'None', not how good the segmentation is. Depth is still fine to evaluate on the test images because the disparity maps are real.

To get a real segmentation score I run the same thing on the 5 validation images, which have proper labels.

## Validation split (real labels)

In [ ]:
val_df, val_maps = run_split(model, "val")
val_df.to_csv(out_dir / "val_log.csv", index=False)
display(val_df.round(4))
print("average:", val_df.drop(columns="image").mean().round(4).to_dict())

# standard per-class IoU from the argmax maps, ignoring 'None' pixels
conf = np.zeros((20, 20), dtype=np.int64)
for p, g in val_maps:
    m = g > 0
    conf += np.bincount(g[m] * 20 + p[m], minlength=400).reshape(20, 20)
inter = np.diag(conf)
union = conf.sum(0) + conf.sum(1) - inter
present = [c for c in range(1, 20) if conf[c].sum() > 0]
per_class = pd.Series({SEG_MAP["classes"][c]: inter[c] / union[c] for c in present}).sort_values()
gt_pixels = pd.Series({SEG_MAP["classes"][c]: int(conf[c].sum()) for c in present})
print(f"mIoU over the {len(present)} classes present: {per_class.mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(per_class.index, per_class.values, color="#2a78d6", height=0.6)
ax.axvline(per_class.mean(), color="#444", lw=1, zorder=0)
ax.text(per_class.mean() + 0.01, len(per_class) - 0.35, f"mIoU {per_class.mean():.3f}", fontsize=9)
ax.set_ylim(-0.6, len(per_class) + 0.2)
for y, v in enumerate(per_class.values):
    ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=8, bbox=dict(facecolor="white", edgecolor="none", pad=0.6))
ax.set_xlim(0, 1.08); ax.set_xlabel("IoU"); ax.set_title("Per-class IoU, validation split")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(out_dir / "per_class_iou.png", dpi=150); plt.show()
print("classes with very few ground-truth pixels:", gt_pixels[gt_pixels < 100].to_dict())

Large classes (road, sky, building, car, vegetation) are around 0.9, while thin or rare ones (pole, traffic light, fence, rider) are much lower. That's why the mIoU is far below the pooled IoU. Truck and terrain have under 100 pixels in these five images, so their 0 is not very meaningful.

## Visual results on the validation images

In [ ]:
ds_val = myData(root_dir=dataset_dir, part="val")
fig, axs = plt.subplots(3, 5, figsize=(18, 7.4))
for r, i in enumerate([0, 2, 4]):
    d = ds_val[i]
    p, g = val_maps[i]
    with torch.no_grad():
        _, pd_ = model(torch.from_numpy(d["rgb"][None]).float())
    row = [d["rgb"].transpose(1, 2, 0)[:, :, [2, 1, 0]], vis_seg(g, SEG_MAP), vis_seg(p, SEG_MAP),
           logdepth(d["dep"][0]), logdepth(pd_[0, 0].numpy())]
    for c, im in enumerate(row):
        axs[r, c].imshow(im.astype(np.uint8)); axs[r, c].axis("off")
for ax, t in zip(axs[0], titles):
    ax.set_title(t)
plt.tight_layout(); plt.savefig(out_dir / "val_results.png", dpi=150); plt.show()

## A bigger test: CamVid (701 images)
Five labelled images are not much, and the full Cityscapes download needs an approved account. CamVid is a public street-scene dataset from Cambridge (UK), filmed with another camera. The SegNet version has 701 labelled images (367 train, 101 val, 233 test). The model was trained on Cityscapes only, so all 701 are new to it and I use all three splits. Its 11 classes map onto the model's classes. The data comes from the SegNet-Tutorial repository (CC BY-NC 4.0): put the `CamVid` folder next to this notebook, or let the next cell download it with git (about 180 MB).

In [ ]:
import subprocess
CAMVID_SPLITS = ["train", "val", "test"]

def camvid_ready(folder):
    return all((folder / s).is_dir() and (folder / (s + "annot")).is_dir() for s in CAMVID_SPLITS)

camvid_dir = next((d for d in [ROOT / "CamVid", ROOT / "SegNet-Tutorial" / "CamVid"] if camvid_ready(d)), None)
if camvid_dir is None:
    repo = ROOT / "SegNet-Tutorial"
    try:   # sparse git download: only the six CamVid image/label folders (about 180 MB)
        if not repo.exists():
            subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                            "https://github.com/alexgkendall/SegNet-Tutorial", str(repo)], check=True)
        subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set"] +
                       [f"CamVid/{s}{a}" for s in CAMVID_SPLITS for a in ["", "annot"]], check=True)
        camvid_dir = repo / "CamVid"
    except Exception as e:
        print("Could not download CamVid automatically:", e)
        camvid_dir = ROOT / "CamVid"   # use whatever splits are there (for example only test/)
splits_found = [s for s in CAMVID_SPLITS if (camvid_dir / s).is_dir() and (camvid_dir / (s + "annot")).is_dir()]
print("CamVid folder:", camvid_dir, "| splits found:", splits_found)

In [ ]:
CAMVID = ["Sky", "Building", "Pole", "Road", "Pavement", "Tree", "SignSymbol", "Fence", "Car", "Pedestrian", "Bicyclist"]  # 11 = unlabelled
# the model's 20 classes -> CamVid's 11 classes (-1 = 'None')
our_to_camvid = np.array([-1, 3, 4, 1, 1, 7, 2, 6, 6, 5, 5, 0, 9, 10, 8, 8, 8, 8, 10, 10])

def camvid_conf(pred_our, gt_cv):
    pc = our_to_camvid[pred_our]
    pc[pc < 0] = 11                       # predicted 'None' -> wrong for every real class
    v = (gt_cv >= 0) & (gt_cv < 11)
    return np.bincount(gt_cv[v].astype(int) * 12 + pc[v], minlength=132).reshape(11, 12)

def iou_11(c):
    inter = np.diag(c[:, :11])
    union = c[:, :11].sum(0) + c.sum(1) - inter
    return inter / union

def boot_interval(confs, iou_fn, n_boot=2000, seed=0):
    # 95% bootstrap interval of the mIoU, resampling whole images
    rng = np.random.default_rng(seed)
    confs = np.stack(confs)
    n = len(confs)
    vals = [np.nanmean(iou_fn(confs[rng.integers(0, n, n)].sum(0))) for _ in range(n_boot)]
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

camvid_confs, camvid_split = [], []
bottom_none, other_none, bottom_road = [], [], []
with torch.no_grad():
    for split in splits_found:
        for f in sorted(glob.glob(str(camvid_dir / split / "*.png"))):
            img = cv2.imread(f)[60:300]                        # crop 480x360 -> 480x240 (2:1 like Cityscapes)
            gt = cv2.imread(str(camvid_dir / (split + "annot") / os.path.basename(f)), cv2.IMREAD_UNCHANGED)[60:300]
            x = cv2.resize(img, (512, 256), interpolation=cv2.INTER_NEAREST)
            s, _ = model(torch.from_numpy(x.transpose(2, 0, 1)[None].copy()).float())
            p = s.argmax(1)[0].numpy()
            bottom_none.append((p[240:] == 0).mean())          # bottom 16 rows predicted as 'None'
            other_none.append((p[:240] == 0).mean())
            bottom_road.append((cv2.resize(gt, (512, 256), interpolation=cv2.INTER_NEAREST)[240:] == 3).mean())
            p = cv2.resize(p.astype(np.uint8), (480, 240), interpolation=cv2.INTER_NEAREST)
            g = gt.astype(int)
            g[g == 11] = -1
            camvid_confs.append(camvid_conf(p, g))
            camvid_split.append(split)

if camvid_confs:
    rows = {}
    for split in splits_found + ["all"]:
        cs = [c for c, s in zip(camvid_confs, camvid_split) if split in (s, "all")]
        lo, hi = boot_interval(cs, iou_11)
        rows[split] = {"images": len(cs), "mIoU": np.nanmean(iou_11(sum(cs))), "95% low": lo, "95% high": hi}
    display(pd.DataFrame(rows).T.round(3))

    camvid_iou = iou_11(sum(camvid_confs))
    n_camvid = len(camvid_confs)
    # Cityscapes validation on the same 11 classes, for a fair comparison
    city_conf = np.zeros((11, 12), dtype=np.int64)
    for p, g in val_maps:
        city_conf += camvid_conf(p, our_to_camvid[g])
    city_iou = iou_11(city_conf)
    share = lambda c: c.sum(1) / c.sum()       # share of labelled pixels per class
    display(pd.DataFrame({"IoU Cityscapes val": city_iou, f"IoU CamVid ({n_camvid})": camvid_iou,
                          "pixel share Cityscapes %": 100 * share(city_conf),
                          "pixel share CamVid %": 100 * share(sum(camvid_confs))}, index=CAMVID).round(3))
    print(f"mIoU: Cityscapes val {city_iou.mean():.3f} | CamVid {camvid_iou.mean():.3f}")
    print(f"CamVid bottom 16 rows: {100 * np.mean(bottom_road):.1f}% road in the labels, but {100 * np.mean(bottom_none):.1f}% "
          f"predicted 'None' (rest of the image: {100 * np.mean(other_none):.1f}%)")
    print(f"Cityscapes val labels, bottom 16 rows: {100 * np.mean([(g[240:] == 0).mean() for _, g in val_maps]):.1f}% 'None' (bonnet + image border)")
else:
    print("No CamVid images found: put the CamVid folder next to this notebook (or install git) to run this part.")

On all 701 CamVid images, from another country and camera, the model still gets an mIoU of about 0.52 (95% interval about 0.51-0.52), and the three splits agree (about 0.50-0.54). It is lower than on the Cityscapes images (about 0.61 for the same classes): the large classes lose 0.12-0.25 IoU, while fence, pedestrian and pavement do better, probably because they are 2-6 times more common in CamVid (see the pixel shares).

The bottom rows are interesting. In CamVid they are mostly road, but the model calls about a third of them 'None', against about 6% in the rest of the image. In the Cityscapes validation labels 94% of those rows are 'None' (the car bonnet and the image border), so the model learned *where* the bonnet usually is, the same way the depth head learned the missing-depth strip on the left edge.

## Another camera and another car: comma10k (1,000 images)
comma10k is an open dataset from comma.ai: 10,000 dash-cam frames from their cars, labelled by volunteers (MIT licence, no academic-only restriction). It has 5 classes: road, lane markings, undrivable, movable (vehicles and people) and "my car" (the car's own body, like the hood). I use the first 1,000 frames of the checked list (0000 to 0999). Lane markings count as road, and I group the model's 20 classes the same way: road -> road, the other static classes -> undrivable, people and vehicles -> movable, and 'None' -> my car. The frames are 1164x874, so I take the middle 2:1 band (rows 146-727), like the CamVid crop, and resize it to 512x256.

The next cell downloads only these 1,000 images and masks with git (about 670 MB). Set `N_COMMA` lower for a quicker run.

In [ ]:
N_COMMA = 1000
comma_dir = ROOT / "comma10k"

def comma_count():
    return len(list((comma_dir / "masks").glob("*.png"))) if (comma_dir / "masks").is_dir() else 0

if comma_count() < N_COMMA:
    try:
        if not (comma_dir / ".git").exists():
            subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                            "https://github.com/commaai/comma10k", str(comma_dir)], check=True)
        listing = subprocess.run(["git", "-C", str(comma_dir), "show", "HEAD:files_trainable"],
                                 capture_output=True, text=True, check=True).stdout.split()
        masks = sorted(x for x in listing if x.startswith("masks/"))[:N_COMMA]
        patterns = "".join(f"/{m}\n/imgs/{m[len('masks/'):]}\n" for m in masks)   # each mask + its image
        subprocess.run(["git", "-C", str(comma_dir), "sparse-checkout", "set", "--no-cone", "--stdin"],
                       input=patterns, text=True, check=True)
    except Exception as e:
        print("Could not download comma10k automatically:", e)
print("comma10k masks found:", comma_count())

In [ ]:
COMMA = ["road", "undrivable", "movable", "my car"]
comma_colours = {(32, 32, 64): 0, (0, 0, 255): 0, (96, 128, 128): 1, (102, 255, 0): 2, (255, 0, 204): 3}  # BGR; lane markings -> road
our_to_comma = np.array([3, 0] + [1] * 10 + [2] * 8)   # None -> my car, road -> road, classes 2-11 -> undrivable, 12-19 -> movable

def iou_4(c):
    inter = np.diag(c)
    return inter / (c.sum(0) + c.sum(1) - inter)

comma_confs, hood_pred = [], np.zeros(20)
hood_bottom, hood_above = np.zeros(2), np.zeros(2)   # [hits, all hood pixels] in / above the bottom 16 rows
comma_files = sorted((comma_dir / "masks").glob("*.png"))[:N_COMMA]
with torch.no_grad():
    for f in comma_files:
        img = cv2.imread(str(comma_dir / "imgs" / f.name))[146:728]     # middle 2:1 band, 582 x 1164
        mask = cv2.imread(str(f))[146:728]
        gt = np.full(mask.shape[:2], -1)
        for colour, k in comma_colours.items():
            gt[(mask == colour).all(axis=2)] = k
        x = cv2.resize(img, (512, 256), interpolation=cv2.INTER_NEAREST)
        s, _ = model(torch.from_numpy(x.transpose(2, 0, 1)[None].copy()).float())
        p256 = s.argmax(1)[0].numpy()
        g256 = cv2.resize(np.where(gt >= 0, gt, 255).astype(np.uint8), (512, 256), interpolation=cv2.INTER_NEAREST)
        hood_bottom += [((g256[240:] == 3) & (p256[240:] == 0)).sum(), (g256[240:] == 3).sum()]   # hood called 'None'
        hood_above += [((g256[:240] == 3) & (p256[:240] == 1)).sum(), (g256[:240] == 3).sum()]    # hood called road
        p = cv2.resize(p256.astype(np.uint8), (1164, 582), interpolation=cv2.INTER_NEAREST)
        v = gt >= 0
        comma_confs.append(np.bincount(gt[v] * 4 + our_to_comma[p][v], minlength=16).reshape(4, 4))
        hood_pred += np.bincount(p[gt == 3], minlength=20)

if comma_confs:
    comma_iou = iou_4(sum(comma_confs))
    lo, hi = boot_interval(comma_confs, iou_4)
    print(f"comma10k: {len(comma_confs)} images, mIoU {comma_iou.mean():.3f} (95% interval {lo:.3f}-{hi:.3f})")

    # Cityscapes validation on the same 4 classes: 'my car' = label ID 1 (ego vehicle), other 'None' pixels are ignored
    city4 = np.zeros((4, 4), dtype=np.int64)
    for name, (p, g) in zip(val_df["image"], val_maps):
        ids = cv2.imread(dataset_dir + f"gtFine_trainvaltest/gtFine/val/frankfurt/{name}_gtFine_labelIds.png", cv2.IMREAD_UNCHANGED)
        ids = cv2.resize(ids, (512, 256), interpolation=cv2.INTER_NEAREST)
        g4 = our_to_comma[g]
        g4[g == 0] = -1
        g4[ids == 1] = 3
        v = g4 >= 0
        city4 += np.bincount(g4[v] * 4 + our_to_comma[p][v], minlength=16).reshape(4, 4)
    city4_iou = iou_4(city4)
    display(pd.DataFrame({"Cityscapes val": city4_iou, f"comma10k ({len(comma_confs)})": comma_iou}, index=COMMA).round(3))
    print(f"mIoU: Cityscapes val {city4_iou.mean():.3f} | comma10k {comma_iou.mean():.3f}")
    top = np.argsort(-hood_pred)[:4]
    print("what the model predicts on the comma car's own body:",
          {SEG_MAP["classes"][k]: f"{100 * hood_pred[k] / hood_pred.sum():.1f}%" for k in top})
    print(f"hood in the bottom 16 rows: {100 * hood_bottom[0] / hood_bottom[1]:.1f}% called 'None' | "
          f"hood above them: {100 * hood_above[0] / hood_above[1]:.1f}% called road")
else:
    print("No comma10k frames found: install git (or put the comma10k folder next to this notebook) to run this part.")

In [ ]:
# both bigger tests in one figure
panels = []
if camvid_confs:
    panels.append((CAMVID, city_iou, camvid_iou, f"CamVid, {n_camvid} images"))
if comma_confs:
    panels.append((COMMA, city4_iou, comma_iou, f"comma10k, {len(comma_confs)} images"))
if not panels:
    print("Nothing to plot: no CamVid or comma10k data was found.")
else:
    fig, axs = plt.subplots(1, len(panels), figsize=(11 if len(panels) == 2 else 6.5, 4.4), squeeze=False,
                            gridspec_kw={"width_ratios": [max(len(names), 6) for names, *_ in panels]})
    for ax, (names, a, b, label) in zip(axs[0], panels):
        order = np.argsort(a)
        y = np.arange(len(names))
        ax.barh(y + 0.2, a[order], height=0.38, color="#2a78d6", label="Cityscapes val, 5 images")
        ax.barh(y - 0.2, b[order], height=0.38, color="#eb6834", label=label)
        ax.set_yticks(y); ax.set_yticklabels([names[k] for k in order]); ax.set_xlim(0, 1); ax.set_xlabel("IoU")
        ax.set_title(f"{label}: mIoU {b.mean():.3f} (Cityscapes {a.mean():.3f})", fontsize=10)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.36), ncol=2, frameon=False, fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(out_dir / "bigger_test.png", dpi=150); plt.show()

comma10k is a harder test: another car, another camera and only four coarse classes. The mIoU is about 0.56 (95% interval about 0.55-0.58), against about 0.92 for the same four classes on the Cityscapes images. Road, undrivable and movable work reasonably well, but 'my car' is the weak class (about 0.24, while the Cityscapes bonnet gets about 0.89). The last line shows why: the model calls the comma hood 'None' mostly in the bottom 16 rows, where the Cityscapes bonnet sits, and 'road' higher up, even though this hood is bigger and reaches higher. Together with the CamVid bottom rows (called 'None' although they are road), this shows the model learned where its own bonnet usually is more than what a car's own body looks like.

## Depth with standard metrics, and how sure the numbers are
The notebook only uses MAE for depth, so I also computed the metrics that depth papers usually report, on valid pixels: RMSE, AbsRel (relative error) and delta < 1.25 (share of pixels within 25% of the true depth). With only 5 labelled images I also wanted to see how much the results move between images.

In [ ]:
def depth_metrics(pred, gt):
    v = (gt > 0) & (gt < 1)              # valid pixels only
    p, g = pred[v] * 100, gt[v] * 100     # in metres
    ratio = np.maximum(p / np.maximum(g, 1e-6), g / np.maximum(p, 1e-6))
    return {"MAE_m": np.abs(p - g).mean(), "RMSE_m": np.sqrt(((p - g) ** 2).mean()),
            "AbsRel": (np.abs(p - g) / g).mean(), "delta<1.25_%": 100 * (ratio < 1.25).mean()}

depth_rows, depth_maps = {}, {}
for part in ["val", "test", "train"]:
    ds = myData(root_dir=dataset_dir, part=part)
    rows, maps = [], []
    with torch.no_grad():
        for i in range(len(ds)):
            d = ds[i]
            _, pred_dep = model(torch.from_numpy(np.ascontiguousarray(d["rgb"][None])).float())
            rows.append(depth_metrics(pred_dep[0, 0].numpy(), d["dep"][0]))
            maps.append((pred_dep[0, 0].numpy(), d["dep"][0]))
    depth_rows[part], depth_maps[part] = pd.DataFrame(rows), maps

summary = {}
for name, df in [("val (5, unseen)", depth_rows["val"]), ("test (5, unseen)", depth_rows["test"]),
                 ("val + test (10, unseen)", pd.concat([depth_rows["val"], depth_rows["test"]])),
                 ("train (5, seen)", depth_rows["train"])]:
    summary[name] = df.mean().round(3).astype(str) + " ± " + df.std(ddof=1).round(3).astype(str)
display(pd.DataFrame(summary).T)
print("SD over the 5 validation images:", val_df[["IoU", "MAE", "TM", "MAE_valid_m"]].std(ddof=1).round(4).to_dict())

In [ ]:
# bootstrap over the 5 validation images: how wide is the uncertainty?
confs = []
for p, g in val_maps:
    m = g > 0
    confs.append(np.bincount(g[m] * 20 + p[m], minlength=400).reshape(20, 20))

def miou_of(cs):
    c = sum(cs)
    inter = np.diag(c)
    union = c.sum(0) + c.sum(1) - inter
    present = [k for k in range(1, 20) if c[k].sum() > 0]
    return np.mean([inter[k] / union[k] for k in present])

rng = np.random.default_rng(0)
boot_miou = [miou_of([confs[j] for j in rng.integers(0, 5, 5)]) for _ in range(10000)]
ious = val_df["IoU"].values
boot_iou = [ious[rng.integers(0, 5, 5)].mean() for _ in range(10000)]
print(f"IoU  {ious.mean():.3f}, 95% interval {np.percentile(boot_iou, 2.5):.3f}-{np.percentile(boot_iou, 97.5):.3f}")
print(f"mIoU {miou_of(confs):.3f}, 95% interval {np.percentile(boot_miou, 2.5):.3f}-{np.percentile(boot_miou, 97.5):.3f}")

The validation AbsRel is much higher than the test one, so I checked which pixels cause it.

In [ ]:
for part in ["val", "test"]:
    near_n = n_valid = near_rel = all_rel = far_sq = all_sq = 0
    for p, g in depth_maps[part]:
        p, g = p * 100, g * 100
        v = (g > 0) & (g < 100)
        rel = np.abs(p - g) / np.maximum(g, 1e-6)
        sq = (p - g) ** 2
        near = v & (g < 5)
        near_n += near.sum(); n_valid += v.sum()
        near_rel += rel[near].sum(); all_rel += rel[v].sum()
        far_sq += sq[v & (g > 50)].sum(); all_sq += sq[v].sum()
    print(f"{part}: closer than 5 m = {100 * near_n / n_valid:.1f}% of valid pixels, AbsRel there {near_rel / near_n:.2f}, "
          f"share of all AbsRel {100 * near_rel / all_rel:.1f}% | pixels beyond 50 m give {100 * far_sq / all_sq:.1f}% of the squared error")

So the results are only rough estimates with five labelled images (the IoU interval is roughly 0.83-0.91, the mIoU one 0.43-0.58). Over all 10 unseen images about 84% of depth pixels are within 25% of the true value. The high validation AbsRel comes mostly from a few very close pixels (under 5 m) that the model puts several times too far away, and RMSE is much bigger than MAE because the far pixels have large errors.

## Check 1: why my numbers are slightly different from the lecturer's log
My test numbers were very close to `trained_model/test_log.csv` but not identical. In the original notebook, cell 30 (PREDIKSI) calls the model without `model.eval()`, so BatchNorm is still in training mode and updates its running statistics with that one validation image before the test loop runs. Here I repeat that order on a fresh copy of the model to see if it explains the difference.

In [ ]:
clean = test_df["TM"].values[:4]
lec = lecturer["test_tm"].values

replica = resunet_mtl()
replica.load_state_dict(torch.load(model_path, map_location=device))
replica.to(device, dtype=torch.float)          # still in training mode, like in cell 30
img0 = cv2.imread(dir_rgb + "val/frankfurt/" + sorted(os.listdir(dir_rgb + "val/frankfurt/"))[0])
x0 = torch.from_numpy(np.expand_dims(resize_matrix(img0, resize=4).transpose(2, 0, 1), axis=0)).float()
replica(x0)                                     # the cell-30 forward pass, BatchNorm stats get updated
replica.eval()
rep_df, _ = run_split(replica, "test")

print("largest TM difference, clean run vs log     :", f"{np.abs(clean - lec).max():.6f}")
print("largest TM difference, cell-30 order vs log :", f"{np.abs(rep_df['TM'].values[:4] - lec).max():.1e}")

With the cell-30 order the numbers match the log almost exactly, so that missing `model.eval()` explains the whole difference. The fix is to call `model.eval()` (and use `torch.no_grad()`) before the forward pass in cell 30. All other results in this notebook are from the clean run.

## Check 2: is BGR input a mistake?
`cv2.imread` gives BGR images but the normaliser in the model uses RGB ImageNet values. I wanted to know if this hurts, so I swapped the channels to RGB and ran the validation images again.

In [ ]:
rgb_df, _ = run_split(model, "val", swap_to_rgb=True)
print(f"BGR (as in the notebook): IoU {val_df['IoU'].mean():.4f}  MAE {val_df['MAE'].mean():.4f}")
print(f"RGB (swapped)           : IoU {rgb_df['IoU'].mean():.4f}  MAE {rgb_df['MAE'].mean():.4f}")

RGB is worse on both, so the model was trained on BGR and the notebook's order is the right one for this model.

## Extra experiments (still inference only)
I wanted to see how the model behaves beyond the standard test, without retraining anything:
- **robustness**: darker, brighter, noisy, blurred and heavily JPEG-compressed versions of the validation images;
- **input size**: feeding bigger images than the 512x256 the notebook uses;
- **flip averaging**: averaging each prediction with the prediction for the mirrored image;
- **train images**: the 5 aachen images, to compare seen and unseen data.

In [ ]:
def corrupt(img, kind, rng):
    x = img.astype(np.float32)
    if kind == "darker":   x = x * 0.5
    if kind == "brighter": x = x * 1.5
    if kind == "noise":    x = x + rng.normal(0, 15, x.shape)
    if kind == "blur":     x = cv2.GaussianBlur(img, (7, 7), 0).astype(np.float32)
    if kind == "jpeg":     x = cv2.imdecode(cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 15])[1], 1).astype(np.float32)
    return np.clip(x, 0, 255).astype(np.uint8)


def evaluate(part, kind=None, flip=False, size=None, rng=None, net=None):
    net = model if net is None else net
    ds = myData(root_dir=dataset_dir, part=part)
    conf = np.zeros((20, 20), dtype=np.int64)
    ious, maes, maes_m = [], [], []
    with torch.no_grad():
        for i in range(len(ds)):
            d = ds[i]
            seg_gt = torch.from_numpy(np.ascontiguousarray(d["seg"][None])).float()
            dep_gt = torch.from_numpy(np.ascontiguousarray(d["dep"][None])).float()
            if size:   # run the network on a bigger image, then bring the outputs back to 512x256
                img = cv2.resize(cv2.imread(ds.rgb[i]), size, interpolation=cv2.INTER_NEAREST)
            else:
                img = d["rgb"].transpose(1, 2, 0).copy()
            if kind:
                img = corrupt(img, kind, rng)
            x = torch.from_numpy(img.transpose(2, 0, 1)[None].copy()).float()
            pred_seg, pred_dep = net(x)
            if flip:
                s2, d2 = net(torch.flip(x, [3]))
                pred_seg = (pred_seg + torch.flip(s2, [3])) / 2
                pred_dep = (pred_dep + torch.flip(d2, [3])) / 2
            if size:
                pred_seg = F.interpolate(pred_seg, size=(256, 512), mode="bilinear", align_corners=True)
                pred_dep = F.interpolate(pred_dep, size=(256, 512), mode="bilinear", align_corners=True)
            ious.append(IOU(pred_seg, seg_gt).item())
            maes.append(F.l1_loss(pred_dep, dep_gt).item())
            valid = (dep_gt > 0) & (dep_gt < 1)
            maes_m.append(((pred_dep - dep_gt).abs()[valid].mean() * 100).item())
            p, g = pred_seg.argmax(1)[0].numpy(), seg_gt.argmax(1)[0].numpy()
            m = g > 0
            conf += np.bincount(g[m] * 20 + p[m], minlength=400).reshape(20, 20)
    inter = np.diag(conf)
    union = conf.sum(0) + conf.sum(1) - inter
    present = [c for c in range(1, 20) if conf[c].sum() > 0]
    return {"IoU": np.mean(ious), "mIoU": np.mean([inter[c] / union[c] for c in present]),
            "MAE": np.mean(maes), "MAE_valid_m": np.mean(maes_m)}


rng = np.random.default_rng(0)   # fixed seed so the noise test is repeatable
results = {"baseline": evaluate("val"), "train split (aachen)": evaluate("train")}
for kind in ["darker", "brighter", "noise", "blur", "jpeg"]:
    results[kind] = evaluate("val", kind=kind, rng=rng)
results["flip averaging"] = evaluate("val", flip=True)
results["input 768x384"] = evaluate("val", size=(768, 384))
results["input 1024x512"] = evaluate("val", size=(1024, 512))

extra_df = pd.DataFrame(results).T
extra_df.to_csv(out_dir / "extra_experiments.csv")
display(extra_df.round(4))

What I see:
- **Lighting** barely matters: 50% darker or brighter images lose less than 0.01 IoU.
- **Noise, blur and JPEG** hurt a lot more, and blur also makes the depth clearly worse (about 5.3 m to 8.1 m).
- **Bigger inputs** make things worse, probably because the model was trained on 512x256 images. A higher resolution would need retraining, not just a bigger input.
- **Train images** score much higher than validation (IoU about 0.933 vs 0.872, depth 2.65 m vs 5.28 m). So the aachen images were most likely used in training and the frankfurt ones were not, which means the validation numbers are a fair test.
- **Flip averaging** helps segmentation a bit but makes depth worse. That was strange, so I looked into it below.

### Why does flipping hurt depth?
I checked, column by column, how often the ground truth is stored as 100 m (missing depth) and how often the model predicts 90 m or more, for the original image and for the mirrored image (flipped back).

In [ ]:
ds_val = myData(root_dir=dataset_dir, part="val")
far_orig, far_flip, gt_missing = np.zeros(512), np.zeros(512), np.zeros(512)
err_orig, err_flip, n_valid = np.zeros(512), np.zeros(512), np.zeros(512)
with torch.no_grad():
    for i in range(len(ds_val)):
        d = ds_val[i]
        x = torch.from_numpy(np.ascontiguousarray(d["rgb"][None])).float()
        g = d["dep"][0]
        _, po = model(x)
        _, pf = model(torch.flip(x, [3]))
        po = po[0, 0].numpy()
        pf = torch.flip(pf, [3])[0, 0].numpy()
        v = (g > 0) & (g < 1)
        far_orig += (po >= 0.9).mean(0) / len(ds_val)
        far_flip += (pf >= 0.9).mean(0) / len(ds_val)
        gt_missing += (g >= 1).mean(0) / len(ds_val)
        err_orig += (np.abs(po - g) * v).sum(0)
        err_flip += (np.abs(pf - g) * v).sum(0)
        n_valid += v.sum(0)

fig, ax = plt.subplots(figsize=(8, 3.2))
cols = np.arange(512)
ax.fill_between(cols, 0, 100 * gt_missing, color="#dcdbd7", lw=0, label="ground truth stored as 100 m")
ax.plot(cols, 100 * far_orig, color="#2a78d6", lw=1.5, label="predicted, original image")
ax.plot(cols, 100 * far_flip, color="#eb6834", lw=1.5, label="predicted, mirrored image")
ax.set_xlim(0, 511); ax.set_ylim(0, 100)
ax.set_xlabel("image column (left to right)"); ax.set_ylabel("% of pixels at 100 m")
ax.legend(loc="upper center", frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(out_dir / "depth_edge.png", dpi=150); plt.show()

print(f"left 32 columns : GT stored as 100 m {100 * gt_missing[:32].mean():.1f}% | predicted 90 m+ {100 * far_orig[:32].mean():.1f}%")
print(f"right 32 columns: depth error original {100 * err_orig[-32:].sum() / n_valid[-32:].sum():.2f} m | mirrored {100 * err_flip[-32:].sum() / n_valid[-32:].sum():.2f} m")

This explains it. The stereo camera has no depth along the left edge, and `disp2dep` stores that missing depth as 100 m, so during training the model learned to predict about 100 m for most of the left strip, whatever the image shows. When I mirror the image, that band moves to the right edge, where the real depth is small, and the error there jumps from about 3.7 m to 71.7 m. The fix would be to leave missing pixels out of the loss (and out of the MAE).

### The 2-epoch model from the notebook's own training run
The course folder also has `resunet_mtl_0/model.pth`, saved by the notebook's training cells after 2 epochs. Those cells train on the val split (frankfurt), so for this model the val images are *seen*, and the train (aachen) and test (berlin) images are *unseen*. I compare it with `best_model.pth`.

In [ ]:
small_path = ROOT / "resunet_mtl_0" / "model.pth"
if small_path.exists():
    small = resunet_mtl()
    print(small.load_state_dict(torch.load(small_path, map_location=device)))
    small.eval()
    compare = {}
    for part in ["val", "train", "test"]:
        compare[f"best_model.pth | {part}"] = evaluate(part)
        compare[f"2-epoch model.pth | {part}"] = evaluate(part, net=small)
    display(pd.DataFrame(compare).T.round(4))
else:
    print("resunet_mtl_0/model.pth not found next to the notebook, skipping this comparison")

The 2-epoch model is far from usable. Even on the 5 images it was trained on, the IoU is about 0.05 and the depth error is over 20 m. On the berlin images, which neither model has seen, its depth error is about 27 m against 4.65 m for `best_model.pth`. (Ignore the test IoU and the empty test mIoU: the test labels are placeholders.) Two epochs on five images are nowhere near enough, so the good results above really come from the properly trained `best_model.pth`.

## Summary
- Validation (real labels): IoU about 0.872 (95% interval 0.83-0.91), mIoU 0.461, depth MAE 0.0675 (about 5.3 m on valid pixels), TM about 0.196.
- Depth on all 10 unseen images: MAE about 5.0 m, RMSE about 12 m, AbsRel about 0.27, and about 84% of pixels within 25% of the true depth.
- CamVid (all 701 images, never seen in training): mIoU about 0.517 (95% interval 0.510-0.523) on 11 classes, against about 0.606 on Cityscapes for the same classes.
- comma10k (1,000 dash-cam frames from other cars): mIoU about 0.564 on 4 coarse classes (Cityscapes: 0.923); "my car" is the weak class, and with the CamVid bottom rows it shows the model learned where its own bonnet usually is.
- Test: TM about 1.019, but only because the test labels are placeholders. Depth on the test images is about 4.6 m on valid pixels.
- The lecturer's log is reproduced once the missing `model.eval()` in cell 30 is copied.
- The model handles lighting changes well but not noise, blur or compression, and it only works at the input size it was trained on.
- The depth head memorised the missing-depth strip on the left edge; masking missing pixels in the loss would fix it.
- The notebook's own 2-epoch model (`resunet_mtl_0`) is far from usable (depth error about 27 m on berlin vs 4.65 m), so the results depend on the properly trained `best_model.pth`.
- Thin and rare classes are the weak point. A higher input resolution (with retraining) and a class-balanced loss would probably help.

The full write-up is in `MTL_Inference_Report_Osama_Ghazal_final.pdf`.